# Machine Learning

## 1- Data Preprocessing

Makine öğrenmesi veri ön işleme pratikleri

Amaç:
    1. Eksik veri tespiti, çıkartılması ve uygun değerler ile doldurma
    2. IQR yöntemiyle sayısal sütunlardaki aykırı değerleri tespit etmek
    3. Kategorik verileri label encoding ve one-hot encoding ile dönüştür
    4. Veriyi train, validasyon ve test kümelerine ayır
    5. Sayısal özelliklere standardization ve normalization uygula

In [32]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder,StandardScaler,MinMaxScaler

### Load the dataset

In [33]:
df = pd.read_csv("musteri_verisi_ml_pratik.csv")
df.head()

,yas,maas,sehir,egitim,deneyim_yili,satin_aldi
0,25.0,32000.0,Ankara,Lisans,1.0,Evet
1,31.0,41000.0,Istanbul,Yuksek Lisans,4.0,Hayir
2,28.0,NaN,Izmir,Lisans,2.0,Beklemede
3,45.0,67000.0,Ankara,Doktora,15.0,Hayir
4,NaN,52000.0,Bursa,Lisans,7.0,Evet


### EDA

In [34]:
df.dtypes

yas             float64
maas            float64
sehir            object
egitim           object
deneyim_yili    float64
satin_aldi       object
dtype: object

In [35]:
df.describe()

,yas,maas,deneyim_yili
count,11.000000,11.000000,11.000000
mean,33.000000,47181.818182,5.909091
std,6.292853,10916.209798,4.323298
min,25.000000,32000.000000,1.000000
25%,28.500000,40000.000000,2.500000
50%,31.000000,45000.000000,5.000000
75%,37.000000,54000.000000,8.000000
max,45.000000,67000.000000,15.000000


In [36]:
df.shape

(12, 6)

### Missing Values

In [37]:
df.isnull().sum()

yas             1
maas            1
sehir           0
egitim          1
deneyim_yili    1
satin_aldi      0
dtype: int64

In [38]:
df_dropna = df.dropna()
df_dropna.shape,df.shape

((8, 6), (12, 6))

In [39]:
df_filled = df.copy()
numeric_variables = df_filled.select_dtypes(include = ["float","int"]).columns.to_list()
numeric_variables

['yas', 'maas', 'deneyim_yili']

In [40]:
for col in numeric_variables:
    df_filled[col] = df_filled[col].fillna(df_filled[col].median())


C:\Users\PC\AppData\Local\Temp\ipykernel_24628\2179659215.py:2: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_filled[col] = df_filled[col].fillna(df_filled[col].median())
C:\Users\PC\AppData\Local\Temp\ipykernel_24628\2179659215.py:2: Fut

In [41]:
df_filled["egitim"] =  df_filled["egitim"].fillna(df_filled["egitim"].mode()[0])
df_filled.isnull().sum()

C:\Users\PC\AppData\Local\Temp\ipykernel_24628\4216156013.py:1: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_filled["egitim"] =  df_filled["egitim"].fillna(df_filled["egitim"].mode()[0])


yas             0
maas            0
sehir           0
egitim          0
deneyim_yili    0
satin_aldi      0
dtype: int64

In [42]:
df_filled.head()

,yas,maas,sehir,egitim,deneyim_yili,satin_aldi
0,25.0,32000.0,Ankara,Lisans,1.0,Evet
1,31.0,41000.0,Istanbul,Yuksek Lisans,4.0,Hayir
2,28.0,45000.0,Izmir,Lisans,2.0,Beklemede
3,45.0,67000.0,Ankara,Doktora,15.0,Hayir
4,31.0,52000.0,Bursa,Lisans,7.0,Evet


### Detecting outliers by using IQR

In [43]:
outlier_mask = pd.Series(False,index= df_filled.index)
for col in numeric_variables:
    Q1 = df_filled[col].quantile(0.25)
    Q3 = df_filled[col].quantile(0.75)
    IQR = Q3-Q1
    upper_border = Q3+1.5*IQR
    lower_border = Q1-1.5*IQR
    column_mask = ((df_filled[col]<lower_border)|(df_filled[col]>upper_border))
    outlier_mask = outlier_mask | column_mask
    print(f" Column mask : ",column_mask.sum())
    if column_mask.any():
        print(f"Outliers : ",df_filled.loc[column_mask,col])

print(f"Column has at least one outlier : ",df_filled.loc[outlier_mask])

 Column mask :  0
 Column mask :  0
 Column mask :  1
Outliers :  3    15.0
Name: deneyim_yili, dtype: float64
Column has at least one outlier :      yas     maas   sehir   egitim  deneyim_yili satin_aldi
3  45.0  67000.0  Ankara  Doktora          15.0      Hayir


### Clean the outliers

In [44]:
df_clean = df_filled.loc[~outlier_mask].copy()
df_clean.reset_index(drop=True,inplace= True)

In [45]:
df_clean

,yas,maas,sehir,egitim,deneyim_yili,satin_aldi
0,25.0,32000.0,Ankara,Lisans,1.0,Evet
1,31.0,41000.0,Istanbul,Yuksek Lisans,4.0,Hayir
2,28.0,45000.0,Izmir,Lisans,2.0,Beklemede
3,31.0,52000.0,Bursa,Lisans,7.0,Evet
4,36.0,48000.0,Istanbul,On Lisans,6.0,Beklemede
5,29.0,39000.0,Izmir,Lisans,5.0,Evet
6,41.0,61000.0,Ankara,Yuksek Lisans,11.0,Hayir
7,33.0,45000.0,Bursa,Lisans,5.0,Beklemede
8,27.0,35000.0,Istanbul,Lisans,2.0,Hayir
9,38.0,56000.0,Izmir,Yuksek Lisans,9.0,Evet


In [46]:
df_filled

,yas,maas,sehir,egitim,deneyim_yili,satin_aldi
0,25.0,32000.0,Ankara,Lisans,1.0,Evet
1,31.0,41000.0,Istanbul,Yuksek Lisans,4.0,Hayir
2,28.0,45000.0,Izmir,Lisans,2.0,Beklemede
3,45.0,67000.0,Ankara,Doktora,15.0,Hayir
4,31.0,52000.0,Bursa,Lisans,7.0,Evet
5,36.0,48000.0,Istanbul,On Lisans,6.0,Beklemede
6,29.0,39000.0,Izmir,Lisans,5.0,Evet
7,41.0,61000.0,Ankara,Yuksek Lisans,11.0,Hayir
8,33.0,45000.0,Bursa,Lisans,5.0,Beklemede
9,27.0,35000.0,Istanbul,Lisans,2.0,Hayir


### Label Encoder

In [47]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df_clean["satin_aldi"])
print(f"Target Classes :",label_encoder.classes_)
print(y)

Target Classes : ['Beklemede' 'Evet' 'Hayir']
[1 2 0 1 0 1 2 0 2 1 0]


In [49]:
X = df_clean.drop(columns = ["satin_aldi"],axis = 1)
X = pd.get_dummies(X,columns = ["egitim"],drop_first=True,dtype = int)
print(X)

     yas     maas  ... egitim_On Lisans  egitim_Yuksek Lisans
0   25.0  32000.0  ...                0                     0
1   31.0  41000.0  ...                0                     1
2   28.0  45000.0  ...                0                     0
3   31.0  52000.0  ...                0                     0
4   36.0  48000.0  ...                1                     0
5   29.0  39000.0  ...                0                     0
6   41.0  61000.0  ...                0                     1
7   33.0  45000.0  ...                0                     0
8   27.0  35000.0  ...                0                     0
9   38.0  56000.0  ...                0                     1
10  30.0  43000.0  ...                0                     0

[11 rows x 6 columns]


In [51]:
# Split the data into Validation and test
X_train_val,X_test,y_train_val,y_test = train_test_split(X,y,test_size = 0.2,stratify=y)
X_train,X_val,y_train_,y_val = train_test_split(X_train_val,y_train_val,test_size = 0.4,stratify=y_train_val)

In [53]:
print(f"X_train_val' s shape",X_train_val.shape)
print(f"X_train' s shape",X_train.shape)
print(f"X_test' s shape",X_test.shape)

X_train_val' s shape (8, 6)
X_train' s shape (4, 6)
X_test' s shape (3, 6)


In [54]:
X_train_standard = X_train_val.copy()
X_val_standard = X_val.copy()
X_test_standard = X_test.copy()

In [55]:
# Call the StandardScaler
scaler = StandardScaler()

In [57]:
X_train_standard[numeric_variables] = scaler.fit_transform(X_train_standard[numeric_variables])
X_val_standard[numeric_variables] = scaler.transform(X_val_standard[numeric_variables])
X_test_standard[numeric_variables] = scaler.transform(X_test_standard[numeric_variables])
X_train_standard,X_val_standard

(         yas      maas  ... egitim_On Lisans  egitim_Yuksek Lisans
 1  -0.069505 -0.598929  ...                0                     1
 8  -1.181582 -1.526303  ...                0                     0
 3  -0.069505  1.101257  ...                0                     0
 2  -0.903562  0.019320  ...                0                     0
 10 -0.347524 -0.289804  ...                0                     0
 9   1.876630  1.719506  ...                0                     1
 5  -0.625543 -0.908054  ...                0                     0
 4   1.320591  0.483007  ...                1                     0
 
 [8 rows x 6 columns],
          yas      maas  ... egitim_On Lisans  egitim_Yuksek Lisans
 10 -0.347524 -0.289804  ...                0                     0
 9   1.876630  1.719506  ...                0                     1
 5  -0.625543 -0.908054  ...                0                     0
 1  -0.069505 -0.598929  ...                0                     1
 
 [4 rows x 6 columns]

In [58]:
minmax = MinMaxScaler()
X_train_minmax = X_train_val.copy()
X_val_minmax = X_val.copy()
X_test_minmax = X_test.copy()

In [59]:
X_train_minmax[numeric_variables] = minmax.fit_transform(X_train_minmax[numeric_variables])
X_val_minmax[numeric_variables] = minmax.transform(X_val_minmax[numeric_variables])
X_test_minmax[numeric_variables] = minmax.transform(X_test_minmax[numeric_variables])

In [60]:
X_test,X_test_standard,X_test_minmax

(    yas     maas   sehir  deneyim_yili  egitim_On Lisans  egitim_Yuksek Lisans
 6  41.0  61000.0  Ankara          11.0                 0                     1
 7  33.0  45000.0   Bursa           5.0                 0                     0
 0  25.0  32000.0  Ankara           1.0                 0                     0,
         yas      maas  ... egitim_On Lisans  egitim_Yuksek Lisans
 6  2.710687  2.492317  ...                0                     1
 7  0.486534  0.019320  ...                0                     0
 0 -1.737620 -1.989990  ...                0                     0
 
 [3 rows x 6 columns],
         yas      maas  ... egitim_On Lisans  egitim_Yuksek Lisans
 6  1.272727  1.238095  ...                0                     1
 7  0.545455  0.476190  ...                0                     0
 0 -0.181818 -0.142857  ...                0                     0
 
 [3 rows x 6 columns])